In [1]:
# ============================================================
# CONFIG
# ============================================================
MODE = "C"
EPOCHS = 275

SEEDS = [3]

CFG = {
    "batch_size": 256,
    "lr": 3e-4,
    "temperature": 0.5,
    "proj_dim": 128,
    "img_size": 96,
    "blur_kernel": 23,
    "sigma": 2.0,
    "data_dir": "/kaggle/working/data",
    "save_dir": "/kaggle/working/checkpoints",
}

In [2]:
def set_seed(seed):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [3]:
import os, time, datetime
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.models import resnet18
from torchvision.datasets import STL10
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CFG["data_dir"], exist_ok=True)
os.makedirs(CFG["save_dir"], exist_ok=True)

print("Device:", device)

Device: cuda


In [4]:
unlabeled = STL10(CFG["data_dir"], split="unlabeled", download=True)
print("Unlabeled size:", len(unlabeled))

100%|██████████| 2.64G/2.64G [01:33<00:00, 28.1MB/s] 


Unlabeled size: 100000


In [5]:
class Residual(nn.Module):
    def __init__(self):
        super().__init__()
        k = CFG["blur_kernel"]
        s = CFG["sigma"]

        coords = torch.arange(k) - k//2
        g = torch.exp(-(coords**2)/(2*s**2))
        g /= g.sum()
        kernel = (g[:,None]*g[None,:]).unsqueeze(0).unsqueeze(0)

        self.register_buffer("kernel", kernel)
        self.k = k

    def forward(self,x):
        C = x.shape[1]
        k = self.kernel.expand(C,1,self.k,self.k)
        blur = F.conv2d(x,k,padding=self.k//2,groups=C)

        r = x - blur
        mean = r.mean([1,2,3],keepdim=True)
        std  = r.std([1,2,3],keepdim=True) + 1e-6

        return (r - mean)/std

res_op = Residual().to(device)

In [6]:
augment = T.Compose([
    T.RandomResizedCrop(CFG["img_size"], scale=(0.2,1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

normalize = T.Normalize([0.485,0.456,0.406],
                        [0.229,0.224,0.225])

In [7]:
class SSLDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        img,_ = self.data[idx]

        v1 = augment(img)
        v2 = augment(img)

        if MODE == "A":
            return normalize(v1), normalize(v2)

        elif MODE == "B":
            return v1, v2   # no residual here

        else:  # C, D
            return normalize(v1), normalize(v2), v1, v2

In [8]:
loader = DataLoader(
    SSLDataset(unlabeled),
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=2,
    drop_last=True
)

print("Batches per epoch:", len(loader))

Batches per epoch: 390


In [9]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            *list(resnet18(weights=None).children())[:-1]
        )

    def forward(self,x):
        return self.backbone(x).flatten(1)


class Projector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,CFG["proj_dim"])
        )

    def forward(self,x):
        return self.net(x)


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.proj_o = Projector()
        self.proj_r = Projector()

    def forward(self,x,r):
        f_o = self.enc(x)
        f_r = self.enc(r)
        return self.proj_o(f_o), self.proj_r(f_r), f_o, f_r


model = Model().to(device)

In [10]:
def ntx(z1,z2):
    B = z1.size(0)

    z = torch.cat([z1,z2],0)
    z = F.normalize(z,dim=1)

    sim = z @ z.T / CFG["temperature"]

    mask = torch.eye(2*B,device=z.device).bool()
    sim.masked_fill_(mask,-1e4)

    target = torch.cat([
        torch.arange(B,2*B),
        torch.arange(0,B)
    ]).to(z.device)

    return F.cross_entropy(sim,target)

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
scaler = GradScaler()

ckpt_path = f"{CFG['save_dir']}/{MODE}.pt"
start_epoch = 1

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["opt"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")

/tmp/ipykernel_55/596967519.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [12]:
for SEED in SEEDS:

    print(f"\n===== RUNNING SEED {SEED} =====")

    set_seed(SEED)

    # IMPORTANT: new model + optimizer per seed
    model = Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    scaler = torch.amp.GradScaler("cuda")

    ckpt_path = f"{CFG['save_dir']}/C_seed{SEED}.pt"

    start_time = time.time()

    for epoch in range(1, EPOCHS+1):

        model.train()
        total_loss = 0
        ep_start = time.time()

        for v1, v2, v1_raw, v2_raw in loader:

            v1, v2 = v1.to(device), v2.to(device)
            v1_raw, v2_raw = v1_raw.to(device), v2_raw.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):

                # residuals on GPU
                r1 = res_op(v1_raw)
                r2 = res_op(v2_raw)

                z_o1, z_r1, f_o1, f_r1 = model(v1, r1)
                z_o2, z_r2, f_o2, f_r2 = model(v2, r2)

                # C loss (NO complementarity)
                loss = ntx(z_o1, z_o2) + ntx(z_r1, z_r2)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        # ===== logging =====
        ep_time = time.time() - ep_start

        print(f"[C][Seed {SEED}] Epoch {epoch}/{EPOCHS} | "
              f"loss {total_loss/len(loader):.4f} | "
              f"{ep_time:.1f}s")

        # ===== save checkpoint =====
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "opt": optimizer.state_dict()
        }, ckpt_path)

    # ===== total time =====
    total_time = time.time() - start_time
    h = int(total_time // 3600)
    m = int((total_time % 3600) // 60)
    s = int(total_time % 60)

    print(f"\n===== DONE SEED {SEED} =====")
    print(f"Time: {h}h {m}m {s}s")
    print(f"Saved: {ckpt_path}")


===== RUNNING SEED 3 =====
[C][Seed 3] Epoch 1/275 | loss 9.4729 | 140.3s
[C][Seed 3] Epoch 2/275 | loss 9.1333 | 143.8s
[C][Seed 3] Epoch 3/275 | loss 9.0544 | 144.9s
[C][Seed 3] Epoch 4/275 | loss 9.0099 | 145.3s
[C][Seed 3] Epoch 5/275 | loss 8.9803 | 144.9s
[C][Seed 3] Epoch 6/275 | loss 8.9584 | 144.6s
[C][Seed 3] Epoch 7/275 | loss 8.9378 | 145.9s
[C][Seed 3] Epoch 8/275 | loss 8.9244 | 145.2s
[C][Seed 3] Epoch 9/275 | loss 8.9112 | 144.2s
[C][Seed 3] Epoch 10/275 | loss 8.8998 | 144.3s
[C][Seed 3] Epoch 11/275 | loss 8.8908 | 145.2s
[C][Seed 3] Epoch 12/275 | loss 8.8829 | 143.9s
[C][Seed 3] Epoch 13/275 | loss 8.8760 | 144.8s
[C][Seed 3] Epoch 14/275 | loss 8.8678 | 144.5s
[C][Seed 3] Epoch 15/275 | loss 8.8622 | 144.6s
[C][Seed 3] Epoch 16/275 | loss 8.8543 | 144.6s
[C][Seed 3] Epoch 17/275 | loss 8.8496 | 143.6s
[C][Seed 3] Epoch 18/275 | loss 8.8482 | 144.2s
[C][Seed 3] Epoch 19/275 | loss 8.8433 | 144.6s
[C][Seed 3] Epoch 20/275 | loss 8.8378 | 144.3s
[C][Seed 3] Epoch 21/

In [13]:
end_time = time.time()
total_time = end_time - start_time

hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)
seconds = int(total_time % 60)

print("\n===== TRAINING COMPLETE =====")
print(f"MODE: {MODE}")
print(f"Final Epoch: {epoch}")
print(f"Final Loss: {total_loss/len(loader):.4f}")
print(f"Total Training Time: {hours}h {minutes}m {seconds}s")
print(f"Checkpoint saved at: {ckpt_path}")


===== TRAINING COMPLETE =====
MODE: C
Final Epoch: 275
Final Loss: 8.6799
Total Training Time: 11h 4m 20s
Checkpoint saved at: /kaggle/working/checkpoints/C_seed3.pt
